Title normalization + seniority extraction (Day 7).
Reads:  jobmarket.silver.stg_postings_api, stg_postings_kaggle
Writes: same tables, rebuilt with title_norm + seniority columns
Method: test on 20 real titles first, then apply to full tables.

In [0]:
from pyspark.sql import functions as F

api    = spark.table("jobmarket.silver.stg_postings_api")
kaggle = spark.table("jobmarket.silver.stg_postings_kaggle")

# union just the title column from both sources
titles = api.select("title_raw").union(kaggle.select("title_raw"))

# random sample of distinct titles
titles.select("title_raw").distinct().sample(0.001).limit(15).display()

# deliberately nasty: titles with punctuation, digits, or ALL CAPS
(titles.filter(
    F.col("title_raw").rlike(r"[()\[\]/|,-]|II|III| SR|Sr\.|JR|\d")
).distinct().sample(0.01).limit(15).display())

In [0]:
def normalize_title(col):
    """Raw title -> normalized title. Column expression in, column expression out."""
    t = F.lower(F.trim(col))

    # 1. kill parenthetical/bracketed segments: "(remote)", "[hybrid]", "(m/f/d)"
    t = F.regexp_replace(t, r"\(.*?\)|\[.*?\]", " ")

    # 2. comma/pipe cut unchanged
    t = F.regexp_replace(t, r"[,|].*$", " ")
    # dash cut ONLY when the final segment is recognizable noise
    noise = (r"\s[-–]\s(remote|hybrid|on[- ]?site|full[- ]?time|part[- ]?time|"
             r"contract|temporary|prn|days?|nights?|weekends?|"
             r"[a-z ]*\b(am|pm|shift)\b[a-z ]*)\s*$")
    t = F.regexp_replace(t, noise, " ")

    # 3. strip seniority tokens (extracted separately BEFORE this runs)
    t = F.regexp_replace(
        t, r"\b(sr\.?|senior|jr\.?|junior|lead|principal|staff|entry[- ]level|intern(ship)?)\b", " ")

    # 4. strip level suffixes: ii, iii, iv, 1-5 as standalone tokens
    t = F.regexp_replace(t, r"\b(i{1,3}v?|[1-5])\b", " ")

    # 4b. remove ordinal tokens entirely: 1st, 2nd, 3rd, 4th...
    t = F.regexp_replace(t, r"\b\d+(st|nd|rd|th)\b", " ")

    # 5. expand common abbreviations (word boundaries — the \beng\b lesson)
    t = F.regexp_replace(t, r"\beng\b", "engineer")
    t = F.regexp_replace(t, r"\bmgr\b", "manager")
    t = F.regexp_replace(t, r"\bdev\b", "developer")
    t = F.regexp_replace(t, r"\bml\b", "machine learning")
    t = F.regexp_replace(t, r"\bbi\b", "business intelligence")
    t = F.regexp_replace(t, r"\bqa\b", "quality assurance")

    # 6. remove leftover punctuation, collapse whitespace, trim
    t = F.regexp_replace(t, r"[^a-z ]", " ")
    t = F.regexp_replace(t, r"\s+", " ")
    return F.trim(t)

In [0]:
def extract_seniority(col):
    """Raw title -> seniority tier. Priority-ordered cascade."""
    t = F.lower(col)
    return (
        F.when(t.rlike(r"\b(principal|staff|distinguished|leader)\b"), "lead")
         .when(t.rlike(r"\blead\b"),                            "lead")
         .when(t.rlike(r"\b(sr\.?|senior|iii|iv)\b"),           "senior")
         .when(t.rlike(r"\bii\b"),                              "mid")
         .when(t.rlike(r"\b(jr\.?|junior|entry|intern(ship)?|graduate|trainee)\b"), "junior")
         .otherwise(F.lit(None))
    )

In [0]:
test = titles.distinct().sample(0.002).limit(25).select(
    "title_raw",
    normalize_title(F.col("title_raw")).alias("title_norm"),
    extract_seniority(F.col("title_raw")).alias("seniority"),
)
display(test)

In [0]:
for table in ["jobmarket.silver.stg_postings_api",
              "jobmarket.silver.stg_postings_kaggle"]:
    df = (spark.table(table)
          .withColumn("title_norm", normalize_title(F.col("title_raw")))
          .withColumn("seniority",  extract_seniority(F.col("title_raw"))))
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(table)

In [0]:
k = spark.table("jobmarket.silver.stg_postings_kaggle")

# (a) compression ratio
raw_n  = k.select("title_raw").distinct().count()
norm_n = k.select("title_norm").distinct().count()
print(f"Kaggle: {raw_n} raw -> {norm_n} normalized "
      f"({(1 - norm_n/raw_n)*100:.1f}% compression)")

# (b) top 20 normalized titles
k.groupBy("title_norm").count().orderBy(F.desc("count")).limit(20).display()

# (c) seniority distribution
k.groupBy("seniority").count().orderBy(F.desc("count")).display()

# (d) agreement matrix vs LinkedIn's labels
(k.filter(F.col("seniority").isNotNull() & F.col("experience_level").isNotNull())
  .groupBy("seniority", "experience_level").count()
  .orderBy("seniority", F.desc("count")).display())

In [0]:
# compression ratio — Cell 6(a), if you didn't capture the print
raw_n  = k.select("title_raw").distinct().count()
norm_n = k.select("title_norm").distinct().count()
print(f"{raw_n} raw -> {norm_n} normalized ({(1 - norm_n/raw_n)*100:.1f}% compression)")

# corruption count
titles = spark.table("jobmarket.silver.stg_postings_api").select("title_raw") \
    .union(spark.table("jobmarket.silver.stg_postings_kaggle").select("title_raw"))
print("Masked titles:", titles.filter(F.col("title_raw").contains("**")).count())